# 19 · L25-29 gap fill — the mask's crossover, per layer

Every analysis so far reads activations at two bands (MID 16-24, LATE 30-34) and skips L25-29 —
yet the mask (Exp 5's `r_bin` sign flip, Exp 7's covert-trait transport demotion, Exp 8's
desirability revaluation) is applied exactly *between* those bands. The probe (06b), shift
vectors (06c), desirability vectors (04) and the Jacobian lenses (10) all already cover 16-34
continuously — only the **per-item activations** were never captured at 25-29.

This notebook:
1. **Gap-fills the activation caches** (`acts_items_<organism>.npz` gains `L25..L29` keys;
   existing layers are kept, one forward pass over the 652 items per organism, ~10 min each on L4).
2. **Exp 5 at full resolution** — reruns the probe-layer table over all 19 layers and overwrites
   `exp5_probe_layers.json` (same schema, superset of layers). Shows *where* `r_bin` crosses zero.
3. **Exp 7 per layer** (no band averaging) → `exp7_signed_transport_perlayer.json`, plus the
   per-layer spearman(cos_z, Exp 6 divergence) over the dark sub-traits — the mask-emergence
   curve. The band-aggregated `exp7_signed_transport.json` is left untouched.
4. **Exp 8 per layer** — the desirability-axis projection of the Exp 6 items at every layer →
   `exp8_desirability_perlayer.json`. Shows where the desirable→undesirable revaluation happens.

What to look for: is the flip **sharp** (one or two layers between 25 and 29 do the work — a
localizable mask circuit) or **gradual**? And does `ref_residual`'s gain stay ~1x through 25-29
("never enters J-space" fully licensed) or **transiently rise** (content surfaces, then is
scrubbed — a stronger mechanism, report it as such)?

**Defaults to the v1 organisms** (`RUN_TAG = "_v1"`, matching the existing
`item_acts_v1_v1` / `components_v1_v1` artifacts). For the -2 retrain set `RUN_TAG = ""` and
swap the `hf` ids in the config cell. With `RUN_TAG = "_v1"` the dark/clinical lenses load from
the Drive `jacobian_lenses/` copies (the HF repo holds the -2 lenses); the base lens is
organism-independent and always comes from HF.

**Hardware:** any GPU >= 20 GB for the three activation passes; the transport SVD-free matmuls
run anywhere.

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
%pip install -q -U "numpy>=2.1" "scipy>=1.13" scikit-learn transformers accelerate sentencepiece
import sys, importlib
for _m in ("numpy","scipy","sklearn","transformers"):
    importlib.import_module(_m); print(_m, "->", getattr(sys.modules[_m], "__version__", "ok"))

In [ ]:
import pathlib
DRIVE = mount_drive()
use_probe_repo()
RUN_TAG = "_v1"   # "_v1" = old organisms (item_acts_v1_v1 / components_v1_v1 — the current
                  # paper artifacts). "" = the -2 retrain (shares 16/17/18's untagged dirs).
DIRS  = (DRIVE / "directions_v1")             if DRIVE else pathlib.Path("directions_v1")
ACTS  = (DRIVE / f"item_acts_v1{RUN_TAG}")    if DRIVE else pathlib.Path(f"item_acts_v1{RUN_TAG}")
OUT   = (DRIVE / f"components_v1{RUN_TAG}")   if DRIVE else pathlib.Path(f"components_v1{RUN_TAG}")
for p in (ACTS, OUT): p.mkdir(parents=True, exist_ok=True)

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass

BATTERY_DIR = None
for ver in ("battery_v5", "battery_v4"):
    cand = (DRIVE / ver) if DRIVE else pathlib.Path(ver)
    if (cand / "rows_dark.csv").exists():
        BATTERY_DIR = cand
        if ver == "battery_v4" and not RUN_TAG:
            print("!! battery_v4 rows = old-organism scores; fine for RUN_TAG='_v1', not for -2.")
        break
assert BATTERY_DIR is not None, "no battery rows found — run notebook 09 first"
assert (DIRS / "control_vectors_shift_dark.pkl").exists(), "shift vectors missing — run 06c"
assert (DIRS / "probe_dark_all.npz").exists(), "probe missing — run 06b"
assert (OUT / "exp6_probe_binary_divergence.json").exists(), \
    "exp6 JSON missing from " + str(OUT) + " — run 17 (or 16) first"
print("directions <-", DIRS, "| battery <-", BATTERY_DIR, "| acts ->", ACTS, "| out ->", OUT)

## 2. Config
`ACT_LAYERS` is the **full 16-34 range** — closing the 25-29 hole is the point. A third band
`gap (25-29)` joins the band summaries so mid/gap/late means print side by side.

In [ ]:
ORGANISMS = [
    {"name": "base",                "hf": "Qwen/Qwen3-8B"},
    {"name": "dark",                "hf": "Koalacrown/dark-qwen3-8b-rl-merged"},       # -2: Koalacrown/dark-2-qwen3-8b
    {"name": "clinical-depression", "hf": "Koalacrown/clinical-depression-qwen3-8b"},  # -2: Koalacrown/clinical-2-qwen3-8b
]
ACT_LAYERS = list(range(16, 35))
PROBE_L    = 18
SELECTOR   = "task_mean"
BATCH      = 16
MAXTOK     = 512
BANDS      = {"mid (16-24)": range(16, 25), "gap (25-29)": range(25, 30),
              "late (30-34)": range(30, 35)}
print(f"{len(ORGANISMS)} organisms | layers {ACT_LAYERS[0]}..{ACT_LAYERS[-1]} | selector {SELECTOR}")

## 3. Items + battery scores
Battery items from `data/source_items/*.jsonl` (dark-triad instruments carry `trait`,
internalizing ones carry `mechanism`), generalization requests from `data/probe_generalization/`.
Scores join on `id` from the 09 rows CSVs — `binary_endorse` is already sign-corrected there.

In [ ]:
import json, glob, csv, collections

def load_jsonl(p):
    return [json.loads(l) for l in open(p) if l.strip()]

ITEMS = {}                       # id -> item dict (+ "side": "trait"|"mechanism", "instrument")
for f in sorted(glob.glob("/content/dt_rl/data/source_items/*.jsonl")):
    inst = pathlib.Path(f).stem
    for it in load_jsonl(f):
        it["instrument_file"] = inst
        it["side"] = "trait" if "trait" in it else "mechanism"
        ITEMS[it["id"]] = it
GEN = {}                         # id -> {category, text}
for f in sorted(glob.glob("/content/dt_rl/data/probe_generalization/*.jsonl")):
    for it in load_jsonl(f):
        GEN[it["id"]] = it

ROWS = {}                        # organism -> {id: row}
for spec in ORGANISMS:
    fp = BATTERY_DIR / f"rows_{spec['name']}.csv"
    if fp.exists():
        ROWS[spec["name"]] = {r["id"]: r for r in csv.DictReader(open(fp))}
    else:
        print(f"!! rows_{spec['name']}.csv missing — Exp 1-3 will skip this organism")

# ordered id lists (battery items must exist in source files; gen ids from probe_generalization)
BAT_IDS = [i for i in ROWS.get("dark", ROWS.get("base", {})) if i in ITEMS]
GEN_IDS = [i for i in ROWS.get("dark", ROWS.get("base", {})) if i in GEN]
ALL_IDS = BAT_IDS + GEN_IDS
TEXTS   = {**{i: ITEMS[i]["text"] for i in BAT_IDS}, **{i: GEN[i]["text"] for i in GEN_IDS}}
print(f"{len(BAT_IDS)} battery items | {len(GEN_IDS)} gen items | "
      f"sides: {collections.Counter(ITEMS[i]['side'] for i in BAT_IDS)}")

## 4. Gap-aware activation capture
Loads each organism's existing `acts_items_<name>.npz`, finds which of `ACT_LAYERS` are missing,
captures **only those layers** (same administration as 16/17/18: bare item text, single user
message, `task_mean` pooling, item order taken from the npz itself so rows stay aligned), and
re-saves the merged npz. Organisms with no missing layers are skipped without loading the model.

In [ ]:
import numpy as np, torch, gc
from tqdm.auto import tqdm
from src.models.huggingface_model import HuggingFaceModel

def gap_fill_org(spec):
    name = spec["name"]; fp = ACTS / f"acts_items_{name}.npz"
    assert fp.exists(), f"{fp} missing — run 17/18 first; this notebook only fills layer gaps"
    z = np.load(fp, allow_pickle=True)
    have = {int(k[1:]) for k in z.files if k.startswith("L")}
    need = [L for L in ACT_LAYERS if L not in have]
    if not need:
        print(f"[skip] {name}: layers complete ({sorted(have)})"); return
    ids = [str(i) for i in z["ids"]]
    print(f"[load] {name} <- {spec['hf']} | filling layers {need}")
    model = HuggingFaceModel(spec["hf"], dtype="bfloat16", device="cuda")
    model.tokenizer.padding_side = "left"
    X = {L: [] for L in need}
    for i in tqdm(range(0, len(ids), BATCH), desc=name):
        chunk = ids[i:i+BATCH]
        msgs = []
        for iid in chunk:
            t = TEXTS[iid]
            tok_ids = model.tokenizer(t, add_special_tokens=False).input_ids
            if len(tok_ids) > MAXTOK:
                t = model.tokenizer.decode(tok_ids[:MAXTOK])
            msgs.append([{"role": "user", "content": t}])
        res = model.get_activations_batch(msgs, need, [SELECTOR])
        for L in need:
            X[L].append(np.asarray(res[SELECTOR][L], dtype=np.float16))
    merged = {k: z[k] for k in z.files}
    merged.update({f"L{L}": np.concatenate(X[L]) for L in need})
    np.savez_compressed(fp, **merged)
    print(f"[done] {name}: npz layers now "
          f"{sorted(int(k[1:]) for k in merged if k.startswith('L'))}")
    del model; gc.collect(); torch.cuda.empty_cache()

for spec in ORGANISMS:
    gap_fill_org(spec)

def load_acts(name):
    z = np.load(ACTS / f"acts_items_{name}.npz")
    ids = list(z["ids"])
    idx = {i: j for j, i in enumerate(ids)}
    return {L: z[f"L{L}"].astype(np.float32) for L in ACT_LAYERS}, idx

ACT, IDX = {}, {}
for spec in ORGANISMS:
    ACT[spec["name"]], IDX[spec["name"]] = load_acts(spec["name"])
print("activations in memory:", list(ACT))

## 5. Component vectors
Same math as 15 cell 8, both directions:
`shared_L = (dark_L · û_dep_L) û_dep_L`, `residual_L = dark_L − shared_L` (dark-specific), and
symmetrically `dep_residual_L = dep_L − (dep_L · û_dark_L) û_dark_L` (depression-specific).

In [ ]:
import pickle

def load_shift(name):
    return pickle.load(open(DIRS / f"control_vectors_shift_{name}.pkl", "rb"))["vectors"]["induced_shift"]

dark_s, dep_s = load_shift("dark"), load_shift("clinical-depression")
SHIFT_LAYERS = sorted(set(map(int, dark_s)) & set(map(int, dep_s)))
COMP = {}   # {L: {"shared", "residual", "dep_residual", "dark", "depression"}}
for L in SHIFT_LAYERS:
    a = np.asarray(dark_s[L], np.float32); b = np.asarray(dep_s[L], np.float32)
    u_dep, u_dark = b / np.linalg.norm(b), a / np.linalg.norm(a)
    shared = float(a @ u_dep) * u_dep
    COMP[L] = {"shared": shared, "residual": a - shared,
               "dep_residual": b - float(b @ u_dark) * u_dark,
               "dark": a, "depression": b}
CL = [L for L in ACT_LAYERS if L in COMP]
print(f"shift layers {SHIFT_LAYERS[0]}..{SHIFT_LAYERS[-1]} | usable with acts: {CL}")

## 6. Exp 5 — layer-specific probe construct validity — full 16-34 resolution
06b saved the probe at **every** layer of the 0.45–0.95 band (per-layer `unit`/`w_raw`/`r` in
`probe_dark_all.npz`) — L18 is just the battery's pick. Per layer: cos(probe, dark-specific /
shared), probe-score → willingness on the dark requests, probe-score → binary endorsement on
positively-keyed dark-triad items, each also as a semi-partial controlling the shared projection.

Reading: **MID keeps `cos_residual` / `sr_will` while LATE loses them** → the probe's construct
validity depends on reading *before* J-space absorbs the dark-specific component into the shared
output pathway. **Flat across depth** → probe validity is not tied to the J-space geometry.
(Positively-keyed items only: `probe_raw` scores the raw text, `binary_endorse` is sign-corrected,
so reverse-keyed items would anti-align the two readouts by construction.)\n\n**This run overwrites `exp5_probe_layers.json` with the 19-layer version** (same schema; the old file was the 14-layer subset). The printed table now shows exactly where `r_bin` crosses zero.

In [ ]:
from scipy import stats as st
pz = np.load(DIRS / "probe_dark_all.npz")
p_layers = list(map(int, pz["layers"]))

def zsc(x):
    x = np.asarray(x, float); return (x - x.mean()) / (x.std() + 1e-12)

def proj_scores(org, ids, L, vec):
    u = vec / np.linalg.norm(vec)
    return ACT[org][L][[IDX[org][i] for i in ids]] @ u

def cosv(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def probe_scores(org, ids, L):
    j = p_layers.index(L)
    x = ACT[org][L][[IDX[org][i] for i in ids]]
    return ((x - pz["mean"][j]) / pz["scale"][j]) @ pz["w_raw"][j] + pz["b_raw"][j]

dt_pos = [i for i in BAT_IDS if ITEMS[i]["side"] == "trait"
          and str(ROWS["dark"][i].get("is_filler", "False")) != "True"
          and float(ROWS["dark"][i].get("sign", 1) or 1) > 0
          and ROWS["dark"][i]["binary_endorse"] not in ("", None)]
y_bin = np.array([float(ROWS["dark"][i]["binary_endorse"]) for i in dt_pos])
will = {i: float(ROWS["dark"][i]["willingness"]) for i in GEN_IDS
        if ROWS["dark"][i]["willingness"] not in ("", None)}
dark_req = [i for i in will if GEN[i]["category"] == "dark"]
y_will = np.array([will[i] for i in dark_req])
print(f"{len(dt_pos)} pos-keyed dark-triad items | {len(dark_req)} dark requests\n")

P_USE = [L for L in CL if L in p_layers]
EXP5 = []
print("layer  heldout_r  cos_res   cos_sh   r_will  sr_will|sh    r_bin  sr_bin|sh")
for L in P_USE:
    j = p_layers.index(L)
    w = pz["unit"][j].astype(np.float32)
    row = {"layer": L, "heldout_r_mu": float(pz["r"][j]),
           "cos_residual": cosv(w, COMP[L]["residual"]),
           "cos_shared":   cosv(w, COMP[L]["shared"])}
    for tag, ids, y in (("will", dark_req, y_will), ("bin", dt_pos, y_bin)):
        ps = zsc(probe_scores("dark", ids, L))
        sh = zsc(proj_scores("dark", ids, L, COMP[L]["shared"]))
        row[f"r_{tag}"]  = st.pearsonr(ps, y)[0]
        resid = ps - np.polyval(np.polyfit(sh, ps, 1), sh)
        row[f"sr_{tag}"] = st.pearsonr(zsc(resid), y)[0]
    EXP5.append(row)
    print(f"  L{L:2d} {row['heldout_r_mu']:+9.3f} {row['cos_residual']:+8.3f} {row['cos_shared']:+8.3f} "
          f"{row['r_will']:+8.3f} {row['sr_will']:+11.3f} {row['r_bin']:+8.3f} {row['sr_bin']:+10.3f}")

for bname, rng_ in BANDS.items():
    rs = [r for r in EXP5 if r["layer"] in rng_]
    if rs:
        print(f"  {bname} mean:  " + "  ".join(
            f"{k}={np.mean([r[k] for r in rs]):+.3f}"
            for k in ("cos_residual", "cos_shared", "r_will", "sr_will", "r_bin")))

with open(OUT / "exp5_probe_layers.json", "w") as f:
    json.dump(EXP5, f, indent=2)
print("\nsaved ->", OUT / "exp5_probe_layers.json")

## 7. Exp 7 per layer — the mask-emergence curve
Same directions and lenses as 18, but **no band averaging**: one row per (lens, layer,
direction) → `exp7_signed_transport_perlayer.json`. Then two summaries per lens:
the `cos_z` curve for the reference components + key sub-traits, and per-layer
spearman(cos_z, Exp 6 divergence) over the dark sub-traits — the layer at which that
correlation goes negative is where the covert/overt sorting is imposed.
Watch `ref_residual`'s `gain_rel` through 25-29: flat ~1x = never enters the workspace;
a transient rise = enters and is scrubbed (report whichever you see).

In [ ]:
import torch
from huggingface_hub import hf_hub_download
DEV = "cuda" if torch.cuda.is_available() else "cpu"

DIRSPEC = {
    "machiavellianism": ("dark",                "mach_iv", None,             "covert -> low gain, -cos"),
    "sd3_mach":         ("dark",                "sd3",     "Machiavellianism","covert -> low gain, -cos"),
    "disinhibition":    ("dark",                "tripm",   "disinhibition",  "covert-ish -> -/0 cos"),
    "rivalry":          ("dark",                "narq",    "rivalry",        "mixed"),
    "meanness":         ("dark",                "tripm",   "meanness",       "mixed"),
    "boldness":         ("dark",                "tripm",   "boldness",       "overt -> +cos"),
    "admiration":       ("dark",                "narq",    "admiration",     "overt -> +cos"),
    "npi_grandiosity":  ("dark",                "npi40",   None,             "overt -> +cos"),
    "rumination_brood": ("clinical-depression", "rrs",     "brooding",       "dystonic -> +cos"),
    "rumination_dep":   ("clinical-depression", "rrs",     "depression",     "dystonic -> +cos"),
    "hopelessness":     ("clinical-depression", "bhs",     None,             "dystonic -> +cos"),
    "worry":            ("clinical-depression", "pswq",    None,             "dystonic -> +cos"),
    "dysregulation":    ("clinical-depression", "ders16",  None,             "dystonic -> +cos"),
    "avoidance":        ("clinical-depression", "aaq2",    None,             "dystonic -> +cos"),
}

def spec_ids(inst, sub):
    return [i for i in BAT_IDS
            if ITEMS[i]["instrument_file"] == inst
            and (sub is None or ITEMS[i].get("subscale") == sub)
            and not ITEMS[i].get("reverse_keyed", False)]

SPEC_IDS = {n: spec_ids(inst, sub) for n, (org, inst, sub, _) in DIRSPEC.items()}

def dmean(org, L, ids):
    return ACT[org][L][[IDX[org][i] for i in ids]].mean(0)

DIR7 = {L: {} for L in ACT_LAYERS}
for L in ACT_LAYERS:
    for n, (org, inst, sub, _) in DIRSPEC.items():
        ids = SPEC_IDS[n]
        if len(ids) >= 4 and org in ACT:
            DIR7[L][n] = dmean(org, L, ids) - dmean("base", L, ids)
    if L in COMP:
        for c in ("shared", "residual", "dep_residual"):
            DIR7[L][f"ref_{c}"] = COMP[L][c]
print("directions per layer:", len(DIR7[ACT_LAYERS[0]]))

LENSES7 = {
    "base": ("neuronpedia/jacobian-lens",
             "qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt"),
    "dark": ("Koalacrown/jacobian-lens-organisms", "dark/jacobian_lens.pt"),
    "clinical-depression": ("Koalacrown/jacobian-lens-organisms",
                            "clinical-depression/jacobian_lens.pt"),
}

def load_J(lname, repo, fname):
    local = (DRIVE / "jacobian_lenses" / f"{lname}_jacobian_lens.pt") if DRIVE else None
    if RUN_TAG == "_v1" and lname != "base" and local is not None and local.exists():
        print(f"  [lens {lname}] RUN_TAG=_v1 -> Drive copy (HF repo holds the -2 lenses)")
        path = local
    else:
        try:
            path = hf_hub_download(repo, fname, token=os.environ.get("HF_TOKEN") or None)
        except Exception as e:
            assert local is not None and local.exists(), \
                f"lens {lname}: HF failed ({type(e).__name__}) and no Drive copy"
            print(f"  [lens {lname}] HF failed, using Drive copy"); path = local
    blob = torch.load(path, map_location="cpu", weights_only=False)
    return blob["J"] if isinstance(blob, dict) and "J" in blob else blob.jacobians

rng7 = np.random.default_rng(0)
RAND7 = rng7.standard_normal((64, 4096)).astype(np.float32)
RAND7 /= np.linalg.norm(RAND7, axis=1, keepdims=True)
R_t = torch.tensor(RAND7)

PER7 = []
for lname, (repo, fname) in LENSES7.items():
    J_all = load_J(lname, repo, fname)
    layers = [L for L in ACT_LAYERS if L in J_all]
    missing = [L for L in ACT_LAYERS if L not in J_all]
    if missing:
        print(f"  !! lens {lname}: no Jacobian at layers {missing} — those stay unmeasured")
    print(f"== lens {lname}: layers {layers[0]}..{layers[-1]} ({len(layers)}) ==")
    for L in layers:
        J = J_all[L].float().to(DEV)
        names = list(DIR7[L])
        V = np.stack([DIR7[L][n] / np.linalg.norm(DIR7[L][n]) for n in names])
        JV = (J @ torch.tensor(V).to(DEV).T).T.cpu().numpy()
        JR = (J @ R_t.to(DEV).T).T.cpu().numpy()
        g_null = (JR ** 2).sum(1)
        c_null = (JR * RAND7).sum(1) / (np.linalg.norm(JR, axis=1) + 1e-12)
        gm, cm, cs = float(g_null.mean()), float(c_null.mean()), float(c_null.std() + 1e-12)
        for k, n in enumerate(names):
            g = float((JV[k] ** 2).sum())
            c = float(JV[k] @ V[k] / (np.linalg.norm(JV[k]) + 1e-12))
            PER7.append({"lens": lname, "layer": L, "direction": n,
                         "gain_rel": g / gm, "cos": c, "cos_z": (c - cm) / cs,
                         "null_cos_mean": cm, "null_cos_sd": cs})
        del J
        if DEV == "cuda": torch.cuda.empty_cache()
    del J_all

from scipy.stats import spearmanr

e6g = json.load(open(OUT / "exp6_probe_binary_divergence.json"))["groups"]
div_by_key = {(g["instrument"], g["subscale"].lower()): g["mean_div"] for g in e6g}
def dir_div(n):
    if n not in DIRSPEC: return None
    _, inst, sub, _ = DIRSPEC[n]
    hits = [v for (gi, gs), v in div_by_key.items()
            if gi == inst and (sub is None or gs == sub.lower())]
    return float(np.mean(hits)) if hits else None
DARK_DIRS = [n for n, s in DIRSPEC.items() if s[0] == "dark" and dir_div(n) is not None]

KEY = ["ref_shared", "ref_dep_residual", "ref_residual",
       "machiavellianism", "disinhibition", "boldness", "admiration"]
for lname in LENSES7:
    rows = {(r["layer"], r["direction"]): r for r in PER7 if r["lens"] == lname}
    Ls = sorted({L for (L, _) in rows})
    if not Ls: continue
    print(f"\n===== lens {lname}: cos_z per layer =====")
    print("layer " + " ".join(f"{n[:12]:>13s}" for n in KEY) + "   rho(div)  gain(ref_res)")
    for L in Ls:
        cz = " ".join(f"{rows[(L, n)]['cos_z']:+13.1f}" if (L, n) in rows else " " * 13
                      for n in KEY)
        both = [(rows[(L, n)]["cos_z"], dir_div(n)) for n in DARK_DIRS if (L, n) in rows]
        rho = spearmanr([b[0] for b in both], [b[1] for b in both])[0] if len(both) >= 4 else float("nan")
        gres = rows[(L, "ref_residual")]["gain_rel"] if (L, "ref_residual") in rows else float("nan")
        mark = " <-- GAP" if 25 <= L <= 29 else ""
        print(f"  L{L:2d} {cz}   {rho:+8.2f} {gres:9.2f}x{mark}")

with open(OUT / "exp7_signed_transport_perlayer.json", "w") as f:
    json.dump(PER7, f, indent=1)
print("\nsaved ->", OUT / "exp7_signed_transport_perlayer.json")

## 8. Exp 8 per layer — where the revaluation happens
Project the Exp 6 items' (dark-organism) activations onto the desirability control-vector axis
(04) at every layer, sign-anchored (prosocial/self-worth items = desirable pole, SRP/PHQ-9 =
undesirable), and correlate with the items' probe_z / binary_z / divergence. The band analysis
found r(probe) flipping +0.37 (mid) → −0.31 (late); this locates the flip layer.

In [ ]:
import pickle
from scipy import stats as st

e6 = json.load(open(OUT / "exp6_probe_binary_divergence.json"))
eids  = [it["id"] for it in e6["items"] if it["id"] in IDX["dark"]]
e_by  = {it["id"]: it for it in e6["items"]}
zp  = np.array([e_by[i]["probe_z"]  for i in eids])
zbn = np.array([e_by[i]["binary_z"] for i in eids])
dv  = np.array([e_by[i]["div"]      for i in eids])
rows8 = [IDX["dark"][i] for i in eids]

ANCH_POS = [i for i in ("acme_07", "acme_08", "rses_01", "rses_03") if i in IDX["dark"]]
ANCH_NEG = [i for i in IDX["dark"] if str(i).startswith(("srp_", "phq9_"))]
print(f"{len(eids)} exp6 items | anchors +{len(ANCH_POS)} / -{len(ANCH_NEG)}")

PER8 = []
for tag in ("base", "dark"):
    dirs = pickle.load(open(DIRS / f"control_vectors_desirability_{tag}.pkl", "rb"))["vectors"]["desirability"]
    print(f"\n== desirability axis: {tag} ==")
    print("layer   r(probe)  r(binary)     r(div)")
    for L in ACT_LAYERS:
        if L not in dirs: continue
        a = ACT["dark"][L] - ACT["dark"][L].mean(0)
        d = np.asarray(dirs[L], np.float32); d /= np.linalg.norm(d)
        p = a @ d
        if p[[IDX["dark"][i] for i in ANCH_POS]].mean() < p[[IDX["dark"][i] for i in ANCH_NEG]].mean():
            p = -p
        s = p[rows8]; s = (s - s.mean()) / (s.std() + 1e-12)
        row = {"axis": tag, "layer": L,
               "r_probe": st.pearsonr(s, zp)[0], "r_binary": st.pearsonr(s, zbn)[0],
               "r_div": st.pearsonr(s, dv)[0]}
        PER8.append(row)
        mark = " <-- GAP" if 25 <= L <= 29 else ""
        print(f"  L{L:2d} {row['r_probe']:+9.3f} {row['r_binary']:+10.3f} {row['r_div']:+10.3f}{mark}")

with open(OUT / "exp8_desirability_perlayer.json", "w") as f:
    json.dump(PER8, f, indent=1)
print("\nsaved ->", OUT / "exp8_desirability_perlayer.json")

---
# Done
Three artifacts in the (tagged) `components_v1` dir: `exp5_probe_layers.json` (now 19 layers),
`exp7_signed_transport_perlayer.json`, `exp8_desirability_perlayer.json` — plus the gap-filled
activation caches. The paper reads off three crossover curves at full resolution: where `r_bin`
flips (Exp 5), where the covert/overt transport sorting appears (Exp 7's per-layer rho), and
where the desirability revaluation happens (Exp 8) — sharp = localizable mask circuit, gradual =
distributed filtering; and `ref_residual`'s gain through 25-29 settles "never enters J-space"
vs "enters and is scrubbed".